In [ ]:
# If running in Google Colab, mount Google Drive so the notebook can access files and save models there.
try:
    import google.colab
    from google.colab import drive
    print('Running in Colab — mounting Google Drive to /content/drive')
    drive.mount('/content/drive')
except Exception as e:
    print('Not running in Colab or mount failed:', str(e))

In [ ]:
# from google.colab import drive; drive.mount('/content/drive')  # uncomment for Colab
base_path = './'  # change to 'drive/MyDrive/POC2PROD/' for Colab

# Fine-tune BERT on `title` -> `tag_id`

This notebook: preprocesses `title` text, creates stratified train/val/test splits, handles class imbalance, fine-tunes `bert-base-uncased` to predict `tag_id` for each title, and plots metrics (loss, accuracy, F1) per epoch. Each major step is in its own cell.

In [ ]:
# Install required packages (run once).
# In a notebook cell this uses the environment's pip.
!pip install -q transformers datasets torch scikit-learn matplotlib seaborn tqdm

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# Load dataset
df = pd.read_csv(os.path.join(base_path, 'stackoverflow_posts.csv'))
print('Columns:', list(df.columns))
# Ensure we have `title` and a numeric label column `tag_id`. If `tag_id` doesn't exist, map `tag_name` -> id.
if 'tag_id' not in df.columns:
    if 'tag_name' in df.columns:
        tag_map = {t: i for i, t in enumerate(sorted(df['tag_name'].dropna().unique()))}
        df['tag_id'] = df['tag_name'].map(tag_map)
        print('Created tag_id from tag_name. Number of classes:', len(tag_map))
    else:
        raise ValueError('Dataset must contain either `tag_id` or `tag_name` column.')
# Drop rows with missing title or label
df = df.dropna(subset=['title', 'tag_id']).reset_index(drop=True)
print('Rows after dropna:', len(df))
df.head()

In [ ]:
# Plot class distribution (tag_id counts)
counts = df['tag_id'].value_counts().sort_index()
plt.figure(figsize=(10,4))
sns.barplot(x=counts.index.astype(str), y=counts.values, palette='viridis')
plt.title('Class counts (tag_id)')
plt.xlabel('tag_id')
plt.ylabel('count')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Show top classes
counts.head(20)

In [ ]:
# Tokenizer and tokenization helper
MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 64

def tokenize_texts(texts, max_length=MAX_LEN):
    return tokenizer(texts, truncation=True, padding='max_length', max_length=max_length, return_tensors='pt')

# Quick tokenization test
sample = df['title'].iloc[:3].astype(str).tolist()
print(sample)
print(tokenize_texts(sample)['input_ids'].shape)

In [ ]:
class TitlesDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        enc = self.tokenizer(text, truncation=True, padding='max_length', max_length=self.max_length, return_tensors='pt')
        item = {k: v.squeeze(0) for k,v in enc.items()}
        item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

# Example
ds = TitlesDataset(df['title'].astype(str).tolist()[:5], df['tag_id'].tolist()[:5], tokenizer)
print(len(ds), ds[0].keys())

In [ ]:
# Create stratified train/val/test splits
TEST_SIZE = 0.1
VAL_SIZE = 0.1  # of the remaining after test split
train_val_df, test_df = train_test_split(df, test_size=TEST_SIZE, stratify=df['tag_id'], random_state=SEED)
train_df, val_df = train_test_split(train_val_df, test_size=VAL_SIZE, stratify=train_val_df['tag_id'], random_state=SEED)

print('Train:', len(train_df), 'Val:', len(val_df), 'Test:', len(test_df))
n_classes = int(df['tag_id'].nunique())
print('Number of classes:', n_classes)

# Create datasets
train_ds = TitlesDataset(train_df['title'].astype(str).tolist(), train_df['tag_id'].tolist(), tokenizer)
val_ds = TitlesDataset(val_df['title'].astype(str).tolist(), val_df['tag_id'].tolist(), tokenizer)
test_ds = TitlesDataset(test_df['title'].astype(str).tolist(), test_df['tag_id'].tolist(), tokenizer)

# Handle class imbalance: compute weights and WeightedRandomSampler for training
class_counts = train_df['tag_id'].value_counts().sort_index().values
class_weights = 1.0 / (class_counts + 1e-9)
# Normalize weights so largest is 1 (optional)
class_weights = class_weights / class_weights.max()
print('Class counts (train):', class_counts[:10])
print('Class weights:', class_weights[:10])

# Per-sample weights
sample_weights = train_df['tag_id'].map(lambda x: class_weights[int(x)]).values
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Initialize model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=n_classes)
model.to(DEVICE)
# Prepare optimizer and scheduler
EPOCHS = 3
total_steps = len(train_loader) * EPOCHS
optimizer = AdamW(model.parameters(), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

# Loss with class weights (on device)
weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
criterion = CrossEntropyLoss(weight=weight_tensor)
print(model.config)

In [ ]:
# Training loop with metric logging per epoch
train_history = {'loss': [], 'acc': [], 'f1': []}
val_history = {'loss': [], 'acc': [], 'f1': []}

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    preds = []
    trues = []
    for batch in tqdm(train_loader, desc=f'Train epoch {epoch+1}'):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item() * input_ids.size(0)
        batch_preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        preds.extend(batch_preds.tolist())
        trues.extend(labels.detach().cpu().numpy().tolist())
    epoch_loss = running_loss / len(train_ds)
    epoch_acc = accuracy_score(trues, preds)
    epoch_f1 = f1_score(trues, preds, average='weighted')
    train_history['loss'].append(epoch_loss)
    train_history['acc'].append(epoch_acc)
    train_history['f1'].append(epoch_f1)
    print(f'Epoch {epoch+1} Train loss {epoch_loss:.4f} acc {epoch_acc:.4f} f1 {epoch_f1:.4f}')

    # Validation
    model.eval()
    val_loss = 0.0
    val_preds = []
    val_trues = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Val epoch {epoch+1}'):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            loss = criterion(logits, labels)
            val_loss += loss.item() * input_ids.size(0)
            val_preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
            val_trues.extend(labels.cpu().numpy().tolist())
    val_epoch_loss = val_loss / len(val_ds)
    val_epoch_acc = accuracy_score(val_trues, val_preds)
    val_epoch_f1 = f1_score(val_trues, val_preds, average='weighted')
    val_history['loss'].append(val_epoch_loss)
    val_history['acc'].append(val_epoch_acc)
    val_history['f1'].append(val_epoch_f1)
    print(f'Epoch {epoch+1} Val loss {val_epoch_loss:.4f} acc {val_epoch_acc:.4f} f1 {val_epoch_f1:.4f}')

# Save histories for plotting
history = {'train': train_history, 'val': val_history}

In [ ]:
# Plot loss, accuracy, and F1 per epoch
epochs = list(range(1, len(history['train']['loss'])+1))
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.plot(epochs, history['train']['loss'], label='train')
plt.plot(epochs, history['val']['loss'], label='val')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1,3,2)
plt.plot(epochs, history['train']['acc'], label='train')
plt.plot(epochs, history['val']['acc'], label='val')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1,3,3)
plt.plot(epochs, history['train']['f1'], label='train')
plt.plot(epochs, history['val']['f1'], label='val')
plt.title('F1 (weighted)')
plt.xlabel('Epoch')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Final evaluation on test set
model.eval()
test_preds = []
test_trues = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc='Test'):
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        test_preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
        test_trues.extend(labels.cpu().numpy().tolist())

print('Test accuracy:', accuracy_score(test_trues, test_preds))
print('Test F1 (weighted):', f1_score(test_trues, test_preds, average='weighted'))
print('
print(classification_report(test_trues, test_preds, digits=4))
: 4,
: 5

In [ ]:
# Save the fine-tuned model and tokenizer
OUT_DIR = os.path.join(base_path, 'tbert_finetuned')
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
print('Saved to', OUT_DIR)